##### 네이버 검색 결과화면에서 뉴스 제목과 링크 추출하기

In [51]:
import requests
import pandas as pd
import time
from bs4 import BeautifulSoup

keyword = 'ai'
url =  f'https://search.naver.com/search.naver?ssc=tab.news.all&where=news&sm=tab_jum&query={keyword}'

# requests로 url로 요청해서 응답을 가져옴

response = requests.get(url)

In [60]:
# 요청이 성공했는지 확인 

# 방법1
# if response.status_code == 200:
# 	print("연결 성공!")
# 	print(response.text)
	
#방법2
try :
  response.raise_for_status()
  print("연결 성공!!")
  # 가져온 결과를 콘솔에 출력
  # print(response.text) #확인용

  # 가져온 결과에서 다음을 만족하는 요소들을 가져옴
  # .sds-comps-base-layout a[data-heatmap-target=".tit"]

  soup = BeautifulSoup(response.text, 'lxml')
  links = soup.select('.sds-comps-base-layout a[data-heatmap-target=".tit"]')


  titles = []
  hrefs = []

  #가져온 요소들을 콘솔에 출력
  for link in links:
    # 제목과 링크(href)를 추출해서 콘솔에 출력
    title = link.text
    href = link.get('href')
    # 제목과 링크를 제목 리스트, 링크 리스트에 각각 뒤에 추가
    titles.append(title)
    hrefs.append(href)

  # 확인용
  # print(titles)
  # print(hrefs)

  # 판다스를 이용하여 DataFrame을 생성
  # 추가할 때 열 이름은 title과 href로 지정

  df = pd.DataFrame({
    'title' : titles,
    'href' : hrefs
  })

  # 확인용으로 5개만 출력
  # print(df.head(5))

  # csv 파일로 저장 
  df.to_csv(f'naver_nl_{keyword}.csv', index=False, encoding='utf-8-sig')

  pass
except Exception as e:
  print(f"예외 발생 : {e}")


연결 성공!!


In [53]:
# 위 예제를 이용하여 키워드가 주어졌을 때 뉴스 목록을 가져와서 데이터프레임으로 반환하는
# 함수를 만들고 테스트 하세요

import requests
from bs4 import BeautifulSoup
import pandas as pd

def get_never_news_list(keyword : str, start : int = 1):
  
  url = f'https://search.naver.com/search.naver'

  params = {
    'where' : 'news',
    'query' : keyword,
    'start' : start,
  }
  headers = {
    'User-Agent' : 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36',
    'Referer' : 'http://www.naver.com'
  }

  # 뉴스 검색 화면을 요청
  response = requests.get(url, params = params)

  try :

    # 연결 성공이 아니면 예외 발생
    response.raise_for_status()
    
    # BeautifulSoup 객체 생성
    soup = BeautifulSoup(response.text, 'lxml')

    # 뉴스 제목 링크들을 가져옴
    links_divs = soup.select('.n8_DDlCjHxLYRO50CY50')
    titles, hrefs = [], []

    # 반복문으로 제목과 링크를 추출하여 리스트에 담음
    for link_div in links_divs:
      # 기사 제목 링크 -> 여기에서 기사 제목 추출
      title_link = link_div.select_one('.XEtVZ4N7DI2Pv29xe7f3')
      # 실제 기사 네이버 링크 -> 여기에서 링크 추출 
      href_link = link_div.select_one('.RQNKk8QZaQZble3gmgEj [data-heatmap-target=".nav"]')

      # 리스트에 제목과 링크를 추가
      hrefs.append(href_link.get("href"))
      titles.append(title_link.text)
    
    return pd.DataFrame({
      'title' : titles,
      'href' : hrefs
    })

  except Exception as e:
    print(f"예외 발생 : {e}")
    return None
  
print(get_never_news_list("ai"))
 

                                       title  \
0                [단독] 크래프톤·한화, 피지컬 AI 합작사 설립   
1      한솔PNS, 'AW 2026' 참가…제조산업 특화 AI 플랫폼 소개   
2                 삼성SDI, 피지컬AI용 전고체 배터리 첫 공개   
3          전북에 피지컬AI·방산·수소기지…DH그룹, 1500억원 투자   
4         포천 농가서 AI 항원 검출…닭 4만5000마리 예방적 살처분   
5       [단독]가상환경서 첨단무기 테스트…크래프톤-한화에어로 ‘AI동맹’   
6        크래프톤-한화에어로스페이스, 피지컬 AI 동맹...합작법인 설립   
7               노동행정도 AI로…노동부, 산재 예측 모델 첫 공개   
8             크래프톤이 게임AI 넘어 피지컬AI로 보폭 넓히는 이유   
9  "SK하이닉스, AI 메모리 수요 급증 최대 수혜…목표가 170만원"-KB   

                                                href  
0  https://n.news.naver.com/mnews/article/011/000...  
1  https://n.news.naver.com/mnews/article/001/001...  
2  https://n.news.naver.com/mnews/article/037/000...  
3  https://n.news.naver.com/mnews/article/003/001...  
4  https://n.news.naver.com/mnews/article/015/000...  
5  https://n.news.naver.com/mnews/article/011/000...  
6  https://n.news.naver.com/mnews/article/018/000...  
7  https://n.news.naver.com/mne

In [54]:
print(get_never_news_list('ai', 21))

                                       title  \
0        "AI 못하면 뒤처진다"…중진공, 중소기업 AX 전환 본격 지원   
1        외부칩 대거 도입한 메타, 자체 AI칩도 공개…개발 난항설 불식   
2         AI 공포에 주가 40% 폭락…'18년 경력' CEO 물러난다   
3           포토샵도 필요 없는 시대…AI 압박 속 어도비 CEO 사임   
4           올레드도 바짝 추격한 中…韓 'AI·보급형 올레드'로 승부   
5                직방, 가상오피스 플랫폼에 AI 다국어 기능 도입   
6            경동나비엔, '파인슬리핑 엑스포'에서 AI 숙면기술 선봬   
7  '사고 치는 AI 로봇 걸러낸다'…UST, 피지컬 AI 안전 검증모델 개발   
8          포스코이앤씨, 오티에르·더샵에 'AI 헬스케어 서비스' 도입   
9     한컴, 오픈데이터로더 PDF v2.0 공개…문서 AI 시장 공략 박차   

                                                href  
0  https://n.news.naver.com/mnews/article/015/000...  
1  https://n.news.naver.com/mnews/article/001/001...  
2  https://n.news.naver.com/mnews/article/015/000...  
3  https://n.news.naver.com/mnews/article/015/000...  
4  https://n.news.naver.com/mnews/article/003/001...  
5  https://n.news.naver.com/mnews/article/003/001...  
6  https://n.news.naver.com/mnews/article/421/000...  
7  https://n.news.naver.com/mne

In [55]:
# 뉴스 리스트 DataFrame을 csv로 저장하는 함수 구현
def save_nl(df : pd.DataFrame, keyword : str, start : int = 1):
  if df is None or df.empty:
    return
  df.to_csv(f'naver_nl_{keyword}_{start//10 + 1}.csv', encoding = 'utf-8-sig', index = False)



In [56]:
start = 41
keyword = 'ai'
nl_df = get_never_news_list(keyword, start)
save_nl(nl_df, keyword, start)


In [57]:
# 반복문으로 총 50개의 ai 뉴스 목록을 가져오도록 작업을 하세요.
# 단 작업할 때 time.sleep(2)를 이용하여 2초의 딜레이를 줌

keyword = 'ai'
for page in range(1, 5 + 1):
  start = (page - 1) * 10 + 1
  nl_df = get_never_news_list(keyword, start)
  save_nl(nl_df, keyword, start)
  print(f"네이버 뉴스{keyword} 검색 {page} 페이지완료")
  time.sleep(2)





네이버 뉴스ai 검색 1 페이지완료
네이버 뉴스ai 검색 2 페이지완료
네이버 뉴스ai 검색 3 페이지완료
네이버 뉴스ai 검색 4 페이지완료
네이버 뉴스ai 검색 5 페이지완료


In [58]:
import sys
print(sys.executable)

c:\Users\hi6\anaconda3\python.exe
